# Traffic benchmark: 4. Train the final deployment selector

This notebook produces one deployment selector after the nested evaluation has been reviewed. It is not a further performance evaluation: all eligible benchmark gaps are used for fitting, so notebook 3 remains the only valid source of held-out performance estimates.


## Final-model selection and training scope

One hyperparameter candidate is selected using its mean inner-fold selected nRMSE across the completed outer folds. The final shared multi-output random forest then learns every candidate-method error target from all eligible benchmark gaps. Missing feature values are median-imputed from final-training rows.

The final scale floor is estimated from all final-training gaps. This is appropriate for deployment and intentionally differs from the fold-local scale floors used in confirmatory evaluation.


In [1]:
from pathlib import Path
import os
import sys
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'pyproject.toml').is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('Start Jupyter from inside the repository.')
    PROJECT_ROOT = PROJECT_ROOT.parent

source_root = str(PROJECT_ROOT / 'src')
if source_root not in sys.path:
    sys.path.insert(0, source_root)

from gap_imputation_benchmark.paths import load_local_environment, local_path_or_default, project_relative_path_or_label
load_local_environment(override=True)

if not os.environ.get('TRAFFIC_DATA_DIR'):
    raise RuntimeError("Set TRAFFIC_DATA_DIR in .env or in the current session.")

DATA_DIR = Path(os.environ['TRAFFIC_DATA_DIR'])
BENCHMARK_DIR = local_path_or_default(
    'TRAFFIC_BENCHMARK_DIR', PROJECT_ROOT / 'benchmarks' / 'traffic',
)
RESULTS_DIR = local_path_or_default(
    'TRAFFIC_LODO_RESULTS_DIR', PROJECT_ROOT / 'results' / 'traffic' / 'lodo',
)
ARTIFACT_DIR = local_path_or_default(
    'TRAFFIC_ARTIFACT_DIR', PROJECT_ROOT / 'artifacts' / 'traffic',
)
CONFIG = PROJECT_ROOT / 'configs' / 'traffic_final.toml'
SCRIPT = PROJECT_ROOT / 'scripts' / 'train_final_traffic_selector.py'
ARTIFACT_LOCATION = project_relative_path_or_label(
    ARTIFACT_DIR, fallback_label='configured artifact directory'
)
print(f'Artifact output: {ARTIFACT_LOCATION}')


Artifact output: artifacts/traffic


## Reproducible artifact creation

The command consumes the frozen configuration, completed nested-evaluation tuning table, and benchmark table (plus required external data where applicable). It does not overwrite an existing artifact directory. Use a new configured `TRAFFIC_ARTIFACT_DIR` path for a deliberate retraining run.

The artifact stores the fitted model and feature imputer; its `metadata.json` is human-readable and contains no machine-specific source-data path.


In [2]:
import subprocess

execution_env = os.environ.copy()
execution_env['PYTHONPATH'] = source_root + os.pathsep + execution_env.get('PYTHONPATH', '')
command = [
    sys.executable, str(SCRIPT), '--config', str(CONFIG), '--data-dir', str(DATA_DIR),
    '--benchmark-dir', str(BENCHMARK_DIR), '--lodo-results-dir', str(RESULTS_DIR),
    '--artifact-dir', str(ARTIFACT_DIR),
]
subprocess.run(command, cwd=PROJECT_ROOT, env=execution_env, check=True)
print(f'Selector artifact written to: {ARTIFACT_LOCATION}')


Final Traffic selector saved: traffic
Selector artifact written to: artifacts/traffic


## Artifact review checklist

Before deployment, verify the domain, candidate-method order, feature order, selected hyperparameters and their nested-evaluation provenance, training-row count, training scope, and scale-floor value. These fields must agree with the reviewed benchmark and evaluation outputs.

The selector predicts method errors and chooses the lowest predicted error for a real missing interval. Reconstruction of the missing values remains the responsibility of the selected candidate imputation method.


In [3]:
import json

metadata = json.loads((ARTIFACT_DIR / 'metadata.json').read_text(encoding='utf-8'))
keys = ('domain', 'training_rows', 'training_districts', 'hyperparameters', 'feature_scale_floor')
{key: metadata.get(key) for key in keys}


{'domain': 'traffic',
 'training_rows': 9997,
 'training_districts': [3, 4, 7, 11],
 'hyperparameters': {'n_estimators': 300,
  'min_samples_leaf': 10,
  'max_depth': None,
  'max_features': 1.0},
 'feature_scale_floor': {'value': 2.5,
  'quantile': 0.01,
  'source': 'all final-training district gaps'}}